In [1]:
# Install af3io from pip
!pip install --quiet af3io
!af3io --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 806.1 kB/s eta 0:00:00
af3io, version 0.4


In [2]:
# Shell command has (some) built-in help
!af3io data-fill --help

Usage: af3io data-fill [OPTIONS]

  Read an input JSON, fill in data pipeline strings from matching sequences
  found in files under --data_dir. Files produced under --output_dir can then
  be used as input for AlphaFold 3 inference, skipping the data pipeline step.

  Files under --data_dir can be either plain JSON (_data.json) or compressed
  with gzip (_data.json.gz). Can specify --data_dir multiple times.

  Data pipeline output is matched by sequence, there is no need to keep a
  consistent set of sequence identifiers (across projects/people/labs) and/or
  file names while still sharing data pipeline output.

  For more than a handful of files in --data_dir, use --write-index to pre-
  compute a sequence-JSON lookup table. This avoids excessive I/O from
  repeatedly reading every file under --data_dir. The index is stored in
  .af3io_data_index.json under --data_dir as a plain-text JSON.

  If (some) sequences do not have data pipeline output, use --missing_dir to
  generate input

In [3]:
# Download one 5k pool from the MGen study
!curl -s https://zenodo.org/records/16920556/files/pools_5k.tar?download=1 \
| tar -xvf - --occurrence pools_5k_0040f80.zip
!unzip pools_5k_0040f80.zip pools_5k_0040f80/pools_5k_0040f80_data.json

pools_5k_0040f80.zip
Archive:  pools_5k_0040f80.zip
  inflating: pools_5k_0040f80/pools_5k_0040f80_data.json  


In [4]:
# AlphaFold 3 stores data pipeline output by adding long strings to the input JSON:
# https://github.com/google-deepmind/alphafold3/blob/main/docs/input.md#multiple-sequence-alignment
!af3io input-show pools_5k_0040f80/pools_5k_0040f80_data.json

{
  "dialect": "alphafold3",
  "version": 2,
  "name": "pools_5k_0040f80",
  "sequences": [
    {
      "protein": {
        "id": "B",
        "sequence": "MYFQNSTQLGWWFLAELIGTFILIIFGNGAVAQVNLKKMATSETKAKFLTVALTWGIGVLFGVLTANAIFKGSGHLNPAISLFYAINGSIKSPTALIWPGFVIGILAQFLGAMIAQTTLNFLFWKQLSSTDPQTVLAMHCTSPSVFNITRNFLTEFIATLILIGGVVAASHFLHNNPNSVPPGFMGLWLVAGIIIAFGGATGSAINPARDLGTRIVFQLTPIKNKDANWKYSWIPVIAPLSAGLVLSIIIGFSPAPVL",
        "modifications": [],
        "unpairedMsa": "<15,084 sequences, 5.55 MB, hash: 3ab334>",
        "pairedMsa": "<25,872 sequences, 10.57 MB, hash: 328091>",
        "templates": "<4 templates, 669.38 KB, hash: dacbd6>"
      }
    },
    {
      "protein": {
        "id": "C",
        "sequence": "MAIRIKSTRVGRFVSESVGLGHPDKICDQIADSILDQCLLQSKTSHVACEVFASKNLILIGGEISTSGYVDVVQTAWRILRNLGYNETDFSFLSCINNQSLEINQAVLKNNEINAGDQGITVGYAVNETKQLMPLGVLLAHSFLKQAEKLTKQFDFLKNDMKSQVVLNYSLNQVECEEVLLSIQHTNAISLTELRKVIENNVILPVLNQYGFQDKKPTCLVNPGGSFVLGGPMADTGLTGRKIIVDTYGPYAHHGGGSFSGKDPSKVDRTGAYFAR

In [5]:
# Create monomer input .json for every chain in the original pool
!af3io input-create monomer_jsons/chain_b.json --sequence MYFQNSTQLGWWFLAELIGTFILIIFGNGAVAQVNLKKMATSETKAKFLTVALTWGIGVLFGVLTANAIFKGSGHLNPAISLFYAINGSIKSPTALIWPGFVIGILAQFLGAMIAQTTLNFLFWKQLSSTDPQTVLAMHCTSPSVFNITRNFLTEFIATLILIGGVVAASHFLHNNPNSVPPGFMGLWLVAGIIIAFGGATGSAINPARDLGTRIVFQLTPIKNKDANWKYSWIPVIAPLSAGLVLSIIIGFSPAPVL
!af3io input-create monomer_jsons/chain_c.json --sequence MAIRIKSTRVGRFVSESVGLGHPDKICDQIADSILDQCLLQSKTSHVACEVFASKNLILIGGEISTSGYVDVVQTAWRILRNLGYNETDFSFLSCINNQSLEINQAVLKNNEINAGDQGITVGYAVNETKQLMPLGVLLAHSFLKQAEKLTKQFDFLKNDMKSQVVLNYSLNQVECEEVLLSIQHTNAISLTELRKVIENNVILPVLNQYGFQDKKPTCLVNPGGSFVLGGPMADTGLTGRKIIVDTYGPYAHHGGGSFSGKDPSKVDRTGAYFARFIAKHIVSLGWASECEVSISWVFSKPNPQSITVKCFNTNIQYDEVLINRVVNNYFNWSITKIIDKLKLLDFVKYSDYAVYGHFGNDLSPWEQPTELDKLECLIKNFH
!af3io input-create monomer_jsons/chain_d.json --sequence MRKNRALKRTVLPDPVFNNTLVTRIINVIMKDGKKGLAQRILYGAFEIIEKRTNQQPLTVFEKAVDNVMPRLELKVRRIAGSNYQVPTEVPPDRRIALALRWIVIFANKRNEKTMLERVANEIIDAFNNTGASVKKKDDTHKMAEANKAFAHMRW
!af3io input-create monomer_jsons/chain_e.json --sequence MIKNLVVIESPNKVKTLKQYLPSDEFEIVSTVGHIREMVYKNFGFDENTYTPIWEDWTKNKQKNPKQKHLLSKFEIIKSIKAKASDAQNIFLASDPDREGEAISWHVYDLLDQKDKAKCKRITFNEITKKAVVDALKQPRNIDLNWVESQFARQILDRMIGFRLSRLLNSYLQAKSAGRVQSVALRFLEEREKEIAKFVPRFWWTVDVLLNKENNQKVVCANKSIPLVLREINPELSASLKLDFEAAENVSGIDFLNEASATRFANQLTGEYEVYFIDEPKIYYSSPNPVYTTASLQKDAINKLGWSSKKVTMVAQRLYEGISVNGKQTALISYPRTDSIRISNQFQSECEKYIEKEFGSHYLADKNKLKRHKKDEKIIQDAHEGIHPTYITITPNDLKNGVKRDEFLLYRLIWIRTVASLMADAKTSRTIVRFINQKNKFYTSSKSLLFDGYQRLYEEIKPNTKDELYIDLSKLKIGDKFSFEKISVNEHKTNPPPRYTQASLIEELEKSNIGRPSTYNTMASVNLERGYANLVNRFFYITELGEKVNNELSKHFGNVINKEFTKKMEKSLDEIAENKVNYQEFLKQFWTNFKSDVKLAENSIQKVKKEKELVERDCPKCNQPLVYRYTKRGNEKFVGCSDFPKCKYSEFSNPKPKLTLETLDELCPECNNKLVKRRTKFNAKKTFIGCSNFPNCRFIKKDNAAEFKQ
!af3io input-create monomer_jsons/chain_f.json --sequence MNNLEKTYKTELVNQLQQQLGFSSIMQVPKLTKIVVNMGVGDAIRDNKFLESALNELHLITGQKPVATKAKNAISTYKLRAGQLIGCKVTLRNKKMWSFLEKLIYIALPRVRDFRGLSLRSFDGKGNYTIGIKEQIIFPEIVYDDIKRIRGFDITIVTSTNKDSEALALLRALKMPFVKE
!af3io input-create monomer_jsons/chain_g.json --sequence MAKKSLKVKQSRPNKFSVRDYTRCLRCGRARAVLSHFGVCRLCFRELAYAGAIPGVKKASW
!af3io input-create monomer_jsons/chain_h.json --sequence MTRNDKRRIRHKRIVKKIRLTNLNNRVVLIVIKSLKNISVQAWDFSKNVVLTSSSSLQLKLKNGNKENAKLVGMDIATKLIKLNQKDVVFDTGGSKYHGRIAALAEGARAKGLNF
!af3io input-create monomer_jsons/chain_i.json --sequence MFKNNLRFTSWINQHKFYQLDLSLKTRSIKQIVLTLVFKTLVLGFFGLIVIFPFYLMVVVSFASDERALDTRTPILWPDSWNFDNFSRVLSDGKYLNAIVVNTLVTVLSVLLTLFFTICMGYSFSLRKWKYKKLVWFFFLSVLILPESALLIGQYRIVIVANWNNPNSPLIVLGLIMPFVSSVFSGFMYRTSFEAIPSQLKESALIDGCNGFNYFLKIALPMVKSTSWTVGILTAFSAWNSYLWPLLLLGNRVDLNINLWVLQQGILDANSSDEQIRTLLNLKMSAAILAILPMFIIYFLFHKRIMNAIKNRANTIKG
!af3io input-create monomer_jsons/chain_j.json --sequence MRVKGTNTTRIRRKKWLKQASGSFGTRKASFKAAKQTVIQASKYAYRDRRQKKREFRSLWILRLNAALRAQGMTYSVFINELKKAKIVINRKVLSELAIKEPNKLNLIINTIKKPTNKPTVAKT
!af3io input-create monomer_jsons/chain_k.json --sequence MSEQKRRTIQIAISEDHYEELQKALELLKGTQLPFSTTVEQFVELILSNYVATSNKISSLAKSGFDVASLQQELEKIGNLSGVDDNLKGFLSELLKTSRNGFSNPNKDGKKNDDDNNSSSKS
!af3io input-create monomer_jsons/chain_l.json --sequence MFFLSKYKLFLDCAYKTLNIIILEMKTNAVVDELSIGVEQNLTELAVYYLETMLTKNKLKKSSIKQFYVTIGPGSFTGQRIATIIAKSWCLLYPSCELYALNSLRFQIPYEHGISKISCGNDQNYCGLYSQTTSEIKLISKADFVKLCKANNELPMYENFENIESYSKLLLSNIDHFERIEDPLTLQPIYLKDPVN
!af3io input-create monomer_jsons/chain_m.json --sequence MLIAIWAMTQEGLIGNNNTLPWMIKQELAHFKKTTLFQALLMGRKTYESLPKVFEKRTIFLLSKDQNYRFEEKGSEVKVINDFWPLIKSYQANKEKDLFICGGKSVYEQTINECDQLIVSIIKKKYKGDQFLKVDLSKFVLNEVVEFEEFNVNYYRKKQQ
!af3io input-create monomer_jsons/chain_n.json --sequence MKKRISTIANLVQSFNPKLVYDIGCDHSYLTSYLIKTNQNLTIVNSDISKNALLSNYQKFKNNNNIHFFVSDGFNNLPELNINPKIGVIAGLGGLKIINIISQKENFINRFVIQPQSNLIELRSFLSLNSWDIVNETLVQDREFIYPILVIEKLKKPFKLTKELVILGPKLINFKDKHCLMKHYQCLLRVYQPKQKPSLMDLKIIETLNKIITSYESS
!af3io input-create monomer_jsons/chain_o.json --sequence MVNKSNSLDELLKQIKITEIIQHYGVKIQTKGNSLLALCPFHDDKNPSMSISSSKNIFKCWACNAAGNGIAFIQKHDQLDWKTALKKAIEICGIKLENWNSNLLTKVDPKQKRYWEINNALITYYQTRLKRETNPNGMNYLVEKRKLNKTLIEQFQLGLAFHNEDKYLCESMERYPFINPKIKPSELYLFSKTNQQGLGFFDFNTKKATFQNQIMIPIHDFNGNPVGFSARSVDNINKLKYKNSADHEFFKKGELLFNFHRLNKNLNQLFIVEGYFDVFTLTNSKFEAVALMGLALNDVQIKAIKAHFKELQTLVLALDNDASGQNAVFSLIEKLNNNNFIVEIVQWEHNYKDWDELYLNKGSEQVILQANKRQNLIEYLVSFFKKQQLDQRVITNKIIAFLTKNQTILNDHSFLIFLIKNLVKLLEYSDEKTLYETVLKHKEKLVSKFDNNRFYINTSGHAQPPQELQKTTAALVQTAFEEAVNELWKPEIFAFALIDKRFLVELKQSHLDEVFKECNFNLFDVELFIEKARIYWSENQTANWVGFESVLDQNYLLNNKARLLEIKDIFLDELTCYQANDFQNYLKTFQTLLKQQKQRLKNLKLTL
!af3io input-create monomer_jsons/chain_p.json --sequence MITKLFFHQVGDNKKRLIWYWKLLIIIAVLAIVIYSWIDNFSSFNQFGLNVFINNITSLFTPNLNHEYTLVRFLAQTAFFVTGGSFLGFIFAILFSYWTAFKIQPFYIALPIRLITIVLRAFPVLLFGFLFSNLFNKQLAATLTISWFSFLWNTKYITTFFENSNLKYFFNKKIREGSGFKAFWTTIFLSENERLWLFFLYSLEANFRWTTLLSIFGIGGIGQLIVDPLSIRVQFDLVLIPLVVLITFLIFIEVVVFLLSSFVFEKNSEDLRPILKTTVIEKRKWKRIIFILFIVVLISLSLANLVTIDYRINDAEFLQDFFNQFFQLKSNLFSSNDPNINPILMLVKLTTQAISLISLVVIFSILFGFISCNLFKKRFSISFKILLLFVRVVPSILLFRLLDPLFLEAKTTIILVLLINHGSSYGQLMSINFNKANQNIINNYKNHGMTKGFILWNYLLVENKPNLINITSDAYDSVIRDLILFGSFGGSIIGSRITNFFERAQFDNLGSVTIPLMVYLIAIEIIFLSVRLTRISVFKNYLY
!af3io input-create monomer_jsons/chain_q.json --sequence MEKTSNTSKPLSRSEINKIIAVATGIKEKKIKEIFKYLNTLLLNELVSRSVCILPENLGKLRITIRNARYQKDMQTGEIRHIPPKPLVRYSPSKTIKETAAKVRWKYAD
!af3io input-create monomer_jsons/chain_r.json --sequence MAKIKFFALGGQDERGKNCYVLEIDNDVFIFNVGSLTPTTAVLGVKKIIPDFSWIQENQARVKGIFIGNAITENLGSLEFLFHTVGFFPIYTSSIGASIIKSKINENKLNIARDKLEIHELKPLETIEISNHSITPFKVSSSLPSSFGFALNTDNGYIVFIDDFIVLNDKNIAFENQLNQIIPKLSDNTLLLITGVGLVGRNSGFTTPKHKSLEQLNRIITPAKGRIFVACYDSNAYSVMTLAQIARMQNRPFIIYSQSFVHLFNTIVRQKLFNNTHLNTISIEEINNSTNSIVVLTSPPDKLYAKLFKIGMNEDERIRYRKSDTFIFMTPKVAGYEEIEAQILDDIARNEVSYYNLGREILSIQASDEDMKFLVSSLKPKYIIPTGGLYRDFINFTMVLKQAGAEQNQILILFNGEVLTIENKKLDSKKNELKLNPKCVDSAGLQEIGASIMFERDQMSESGVVIIIIYFDQKKSEFLNEITYSFLGVSLDVPEKDKLKTKMEELIKKQINDIKDFTTIKKRIGKEISKELKVSIKRAVMNLFTKMTSKAPLILSTIISI
!af3io input-create monomer_jsons/chain_s.json --sequence MVLKTKENKKFDIYLKSSDFAVSKKASKLIKKLNKKHPKRKSLNSFEAKKYDIYFKEVCKAVTNGINNQLICNHINLKILPGEFVVILGKSGSGKTSLLSLISALDRPTSGDSFVCGTNTICCSDAKLTALRNKNVGYIFQQYGLLRDLDVDDNIKLALPLKKRFNNNLEELLERLELKEHRHKKVHKLSGGQQQRVAIARALIKEPKILFGDEPTGAVNIDISKKILQFFVEYNRDKGTTIVIVTHNEKIVELAKRVIKIHDGKIIVDYLNQNPKTIEQINWV

Setting name to: chain_b
Write:	/content/monomer_jsons/chain_b.json
Setting name to: chain_c
Write:	/content/monomer_jsons/chain_c.json
Setting name to: chain_d
Write:	/content/monomer_jsons/chain_d.json
Setting name to: chain_e
Write:	/content/monomer_jsons/chain_e.json
Setting name to: chain_f
Write:	/content/monomer_jsons/chain_f.json
Setting name to: chain_g
Write:	/content/monomer_jsons/chain_g.json
Setting name to: chain_h
Write:	/content/monomer_jsons/chain_h.json
Setting name to: chain_i
Write:	/content/monomer_jsons/chain_i.json
Setting name to: chain_j
Write:	/content/monomer_jsons/chain_j.json
Setting name to: chain_k
Write:	/content/monomer_jsons/chain_k.json
Setting name to: chain_l
Write:	/content/monomer_jsons/chain_l.json
Setting name to: chain_m
Write:	/content/monomer_jsons/chain_m.json
Setting name to: chain_n
Write:	/content/monomer_jsons/chain_n.json
Setting name to: chain_o
Write:	/content/monomer_jsons/chain_o.json
Setting name to: chain_p
Write:	/content/monomer

In [6]:
# Eighteen jsons, one per chain in the original pool
!ls -1 monomer_jsons/

chain_b.json
chain_c.json
chain_d.json
chain_e.json
chain_f.json
chain_g.json
chain_h.json
chain_i.json
chain_j.json
chain_k.json
chain_l.json
chain_m.json
chain_n.json
chain_o.json
chain_p.json
chain_q.json
chain_r.json
chain_s.json


In [7]:
# Each file only has the sequence, no data pipeline strings
# The "name" field is inferred from the file name (required to match by AlphaFold 3)
!af3io input-show monomer_jsons/chain_g.json

{
  "dialect": "alphafold3",
  "version": 2,
  "name": "chain_g",
  "sequences": [
    {
      "protein": {
        "id": "A",
        "sequence": "MAKKSLKVKQSRPNKFSVRDYTRCLRCGRARAVLSHFGVCRLCFRELAYAGAIPGVKKASW"
      }
    }
  ],
  "modelSeeds": [
    1
  ],
  "bondedAtomPairs": null,
  "userCCD": null
}


In [8]:
# Copy data pipeline strings using af3io data-fill
# https://github.com/google-deepmind/alphafold3/blob/main/docs/performance.md#pre-computing-and-reusing-msa-and-templates
!af3io data-fill --data_dir pools_5k_0040f80 --input_dir monomer_jsons --output_dir monomer_msas

Create data index from: pools_5k_0040f80
Reading files:  [####################################]  100%                            
Read 18 protein, 0 dna, 0 rna sequence(s)
Data index has 18 protein, 0 dna, 0 rna sequence(s)
Read:	monomer_jsons/chain_b.json
	fill id=A from id=B in /content/pools_5k_0040f80/pools_5k_0040f80_data.json
Write:	monomer_msas/chain_b_data.json
Read:	monomer_jsons/chain_n.json
	fill id=A from id=N in /content/pools_5k_0040f80/pools_5k_0040f80_data.json
Write:	monomer_msas/chain_n_data.json
Read:	monomer_jsons/chain_i.json
	fill id=A from id=I in /content/pools_5k_0040f80/pools_5k_0040f80_data.json
Write:	monomer_msas/chain_i_data.json
Read:	monomer_jsons/chain_j.json
	fill id=A from id=J in /content/pools_5k_0040f80/pools_5k_0040f80_data.json
Write:	monomer_msas/chain_j_data.json
Read:	monomer_jsons/chain_o.json
	fill id=A from id=O in /content/pools_5k_0040f80/pools_5k_0040f80_data.json
Write:	monomer_msas/chain_o_data.json
Read:	monomer_jsons/chain_c.json
	fi

In [9]:
# One example of with MSA; file has been renamed to have the _data suffix
!af3io input-show monomer_msas/chain_g_data.json

{
  "dialect": "alphafold3",
  "version": 2,
  "name": "chain_g",
  "sequences": [
    {
      "protein": {
        "id": "A",
        "sequence": "MAKKSLKVKQSRPNKFSVRDYTRCLRCGRARAVLSHFGVCRLCFRELAYAGAIPGVKKASW",
        "modifications": [],
        "unpairedMsa": "<12,215 sequences, 2.02 MB, hash: a89d58>",
        "pairedMsa": "<26,595 sequences, 5.47 MB, hash: 059bc0>",
        "templates": "<4 templates, 149.64 KB, hash: 6770a1>"
      }
    }
  ],
  "modelSeeds": [
    1
  ],
  "bondedAtomPairs": null,
  "userCCD": null
}


In [10]:
# Typically hundreds/thousands of monomer MSAs, and would be I/O costly to go through all of them
# Solution - create "index" mapping available sequences to data pipeline .jsons
!af3io data-fill --data_dir monomer_msas --write-index

Create data index from: monomer_msas
Reading files:  [####################################]  100%                             
Read 18 protein, 0 dna, 0 rna sequence(s)
Writing index to: monomer_msas/.af3io_data_index.json
Data index has 18 protein, 0 dna, 0 rna sequence(s)


In [11]:
# Index stored as a dictionary in a plain-text .json
!head -n 10 monomer_msas/.af3io_data_index.json

{
  "protein": {
    "MTRNDKRRIRHKRIVKKIRLTNLNNRVVLIVIKSLKNISVQAWDFSKNVVLTSSSSLQLKLKNGNKENAKLVGMDIATKLIKLNQKDVVFDTGGSKYHGRIAALAEGARAKGLNF": "/content/monomer_msas/chain_h_data.json",
    "MLIAIWAMTQEGLIGNNNTLPWMIKQELAHFKKTTLFQALLMGRKTYESLPKVFEKRTIFLLSKDQNYRFEEKGSEVKVINDFWPLIKSYQANKEKDLFICGGKSVYEQTINECDQLIVSIIKKKYKGDQFLKVDLSKFVLNEVVEFEEFNVNYYRKKQQ": "/content/monomer_msas/chain_m_data.json",
    "MKKRISTIANLVQSFNPKLVYDIGCDHSYLTSYLIKTNQNLTIVNSDISKNALLSNYQKFKNNNNIHFFVSDGFNNLPELNINPKIGVIAGLGGLKIINIISQKENFINRFVIQPQSNLIELRSFLSLNSWDIVNETLVQDREFIYPILVIEKLKKPFKLTKELVILGPKLINFKDKHCLMKHYQCLLRVYQPKQKPSLMDLKIIETLNKIITSYESS": "/content/monomer_msas/chain_n_data.json",
    "MEKTSNTSKPLSRSEINKIIAVATGIKEKKIKEIFKYLNTLLLNELVSRSVCILPENLGKLRITIRNARYQKDMQTGEIRHIPPKPLVRYSPSKTIKETAAKVRWKYAD": "/content/monomer_msas/chain_q_data.json",
    "MRVKGTNTTRIRRKKWLKQASGSFGTRKASFKAAKQTVIQASKYAYRDRRQKKREFRSLWILRLNAALRAQGMTYSVFINELKKAKIVINRKVLSELAIKEPNKLNLIINTIKKPTNKPTVAKT": "/content/monomer_msas/chain_j_data.json",
  

In [12]:
# To illustrate, create input file with sequences from the original pool
!af3io input-create --model_seed 4 \
    --type protein --id B --sequence MYFQNSTQLGWWFLAELIGTFILIIFGNGAVAQVNLKKMATSETKAKFLTVALTWGIGVLFGVLTANAIFKGSGHLNPAISLFYAINGSIKSPTALIWPGFVIGILAQFLGAMIAQTTLNFLFWKQLSSTDPQTVLAMHCTSPSVFNITRNFLTEFIATLILIGGVVAASHFLHNNPNSVPPGFMGLWLVAGIIIAFGGATGSAINPARDLGTRIVFQLTPIKNKDANWKYSWIPVIAPLSAGLVLSIIIGFSPAPVL \
    --type protein --id C --sequence MAIRIKSTRVGRFVSESVGLGHPDKICDQIADSILDQCLLQSKTSHVACEVFASKNLILIGGEISTSGYVDVVQTAWRILRNLGYNETDFSFLSCINNQSLEINQAVLKNNEINAGDQGITVGYAVNETKQLMPLGVLLAHSFLKQAEKLTKQFDFLKNDMKSQVVLNYSLNQVECEEVLLSIQHTNAISLTELRKVIENNVILPVLNQYGFQDKKPTCLVNPGGSFVLGGPMADTGLTGRKIIVDTYGPYAHHGGGSFSGKDPSKVDRTGAYFARFIAKHIVSLGWASECEVSISWVFSKPNPQSITVKCFNTNIQYDEVLINRVVNNYFNWSITKIIDKLKLLDFVKYSDYAVYGHFGNDLSPWEQPTELDKLECLIKNFH \
    --type protein --id D --sequence MRKNRALKRTVLPDPVFNNTLVTRIINVIMKDGKKGLAQRILYGAFEIIEKRTNQQPLTVFEKAVDNVMPRLELKVRRIAGSNYQVPTEVPPDRRIALALRWIVIFANKRNEKTMLERVANEIIDAFNNTGASVKKKDDTHKMAEANKAFAHMRW \
    --type protein --id E --sequence MIKNLVVIESPNKVKTLKQYLPSDEFEIVSTVGHIREMVYKNFGFDENTYTPIWEDWTKNKQKNPKQKHLLSKFEIIKSIKAKASDAQNIFLASDPDREGEAISWHVYDLLDQKDKAKCKRITFNEITKKAVVDALKQPRNIDLNWVESQFARQILDRMIGFRLSRLLNSYLQAKSAGRVQSVALRFLEEREKEIAKFVPRFWWTVDVLLNKENNQKVVCANKSIPLVLREINPELSASLKLDFEAAENVSGIDFLNEASATRFANQLTGEYEVYFIDEPKIYYSSPNPVYTTASLQKDAINKLGWSSKKVTMVAQRLYEGISVNGKQTALISYPRTDSIRISNQFQSECEKYIEKEFGSHYLADKNKLKRHKKDEKIIQDAHEGIHPTYITITPNDLKNGVKRDEFLLYRLIWIRTVASLMADAKTSRTIVRFINQKNKFYTSSKSLLFDGYQRLYEEIKPNTKDELYIDLSKLKIGDKFSFEKISVNEHKTNPPPRYTQASLIEELEKSNIGRPSTYNTMASVNLERGYANLVNRFFYITELGEKVNNELSKHFGNVINKEFTKKMEKSLDEIAENKVNYQEFLKQFWTNFKSDVKLAENSIQKVKKEKELVERDCPKCNQPLVYRYTKRGNEKFVGCSDFPKCKYSEFSNPKPKLTLETLDELCPECNNKLVKRRTKFNAKKTFIGCSNFPNCRFIKKDNAAEFKQ \
    --type protein --id F --sequence MNNLEKTYKTELVNQLQQQLGFSSIMQVPKLTKIVVNMGVGDAIRDNKFLESALNELHLITGQKPVATKAKNAISTYKLRAGQLIGCKVTLRNKKMWSFLEKLIYIALPRVRDFRGLSLRSFDGKGNYTIGIKEQIIFPEIVYDDIKRIRGFDITIVTSTNKDSEALALLRALKMPFVKE \
    --type protein --id G --sequence MAKKSLKVKQSRPNKFSVRDYTRCLRCGRARAVLSHFGVCRLCFRELAYAGAIPGVKKASW \
    --type protein --id H --sequence MTRNDKRRIRHKRIVKKIRLTNLNNRVVLIVIKSLKNISVQAWDFSKNVVLTSSSSLQLKLKNGNKENAKLVGMDIATKLIKLNQKDVVFDTGGSKYHGRIAALAEGARAKGLNF \
    --type protein --id I --sequence MFKNNLRFTSWINQHKFYQLDLSLKTRSIKQIVLTLVFKTLVLGFFGLIVIFPFYLMVVVSFASDERALDTRTPILWPDSWNFDNFSRVLSDGKYLNAIVVNTLVTVLSVLLTLFFTICMGYSFSLRKWKYKKLVWFFFLSVLILPESALLIGQYRIVIVANWNNPNSPLIVLGLIMPFVSSVFSGFMYRTSFEAIPSQLKESALIDGCNGFNYFLKIALPMVKSTSWTVGILTAFSAWNSYLWPLLLLGNRVDLNINLWVLQQGILDANSSDEQIRTLLNLKMSAAILAILPMFIIYFLFHKRIMNAIKNRANTIKG \
    --type protein --id J --sequence MRVKGTNTTRIRRKKWLKQASGSFGTRKASFKAAKQTVIQASKYAYRDRRQKKREFRSLWILRLNAALRAQGMTYSVFINELKKAKIVINRKVLSELAIKEPNKLNLIINTIKKPTNKPTVAKT \
    --type protein --id K --sequence MSEQKRRTIQIAISEDHYEELQKALELLKGTQLPFSTTVEQFVELILSNYVATSNKISSLAKSGFDVASLQQELEKIGNLSGVDDNLKGFLSELLKTSRNGFSNPNKDGKKNDDDNNSSSKS \
    --type protein --id L --sequence MFFLSKYKLFLDCAYKTLNIIILEMKTNAVVDELSIGVEQNLTELAVYYLETMLTKNKLKKSSIKQFYVTIGPGSFTGQRIATIIAKSWCLLYPSCELYALNSLRFQIPYEHGISKISCGNDQNYCGLYSQTTSEIKLISKADFVKLCKANNELPMYENFENIESYSKLLLSNIDHFERIEDPLTLQPIYLKDPVN \
    --type protein --id M --sequence MLIAIWAMTQEGLIGNNNTLPWMIKQELAHFKKTTLFQALLMGRKTYESLPKVFEKRTIFLLSKDQNYRFEEKGSEVKVINDFWPLIKSYQANKEKDLFICGGKSVYEQTINECDQLIVSIIKKKYKGDQFLKVDLSKFVLNEVVEFEEFNVNYYRKKQQ \
    --type protein --id N --sequence MKKRISTIANLVQSFNPKLVYDIGCDHSYLTSYLIKTNQNLTIVNSDISKNALLSNYQKFKNNNNIHFFVSDGFNNLPELNINPKIGVIAGLGGLKIINIISQKENFINRFVIQPQSNLIELRSFLSLNSWDIVNETLVQDREFIYPILVIEKLKKPFKLTKELVILGPKLINFKDKHCLMKHYQCLLRVYQPKQKPSLMDLKIIETLNKIITSYESS \
    --type protein --id O --sequence MVNKSNSLDELLKQIKITEIIQHYGVKIQTKGNSLLALCPFHDDKNPSMSISSSKNIFKCWACNAAGNGIAFIQKHDQLDWKTALKKAIEICGIKLENWNSNLLTKVDPKQKRYWEINNALITYYQTRLKRETNPNGMNYLVEKRKLNKTLIEQFQLGLAFHNEDKYLCESMERYPFINPKIKPSELYLFSKTNQQGLGFFDFNTKKATFQNQIMIPIHDFNGNPVGFSARSVDNINKLKYKNSADHEFFKKGELLFNFHRLNKNLNQLFIVEGYFDVFTLTNSKFEAVALMGLALNDVQIKAIKAHFKELQTLVLALDNDASGQNAVFSLIEKLNNNNFIVEIVQWEHNYKDWDELYLNKGSEQVILQANKRQNLIEYLVSFFKKQQLDQRVITNKIIAFLTKNQTILNDHSFLIFLIKNLVKLLEYSDEKTLYETVLKHKEKLVSKFDNNRFYINTSGHAQPPQELQKTTAALVQTAFEEAVNELWKPEIFAFALIDKRFLVELKQSHLDEVFKECNFNLFDVELFIEKARIYWSENQTANWVGFESVLDQNYLLNNKARLLEIKDIFLDELTCYQANDFQNYLKTFQTLLKQQKQRLKNLKLTL \
    --type protein --id P --sequence MITKLFFHQVGDNKKRLIWYWKLLIIIAVLAIVIYSWIDNFSSFNQFGLNVFINNITSLFTPNLNHEYTLVRFLAQTAFFVTGGSFLGFIFAILFSYWTAFKIQPFYIALPIRLITIVLRAFPVLLFGFLFSNLFNKQLAATLTISWFSFLWNTKYITTFFENSNLKYFFNKKIREGSGFKAFWTTIFLSENERLWLFFLYSLEANFRWTTLLSIFGIGGIGQLIVDPLSIRVQFDLVLIPLVVLITFLIFIEVVVFLLSSFVFEKNSEDLRPILKTTVIEKRKWKRIIFILFIVVLISLSLANLVTIDYRINDAEFLQDFFNQFFQLKSNLFSSNDPNINPILMLVKLTTQAISLISLVVIFSILFGFISCNLFKKRFSISFKILLLFVRVVPSILLFRLLDPLFLEAKTTIILVLLINHGSSYGQLMSINFNKANQNIINNYKNHGMTKGFILWNYLLVENKPNLINITSDAYDSVIRDLILFGSFGGSIIGSRITNFFERAQFDNLGSVTIPLMVYLIAIEIIFLSVRLTRISVFKNYLY \
    --type protein --id Q --sequence MEKTSNTSKPLSRSEINKIIAVATGIKEKKIKEIFKYLNTLLLNELVSRSVCILPENLGKLRITIRNARYQKDMQTGEIRHIPPKPLVRYSPSKTIKETAAKVRWKYAD \
    --type protein --id R --sequence MAKIKFFALGGQDERGKNCYVLEIDNDVFIFNVGSLTPTTAVLGVKKIIPDFSWIQENQARVKGIFIGNAITENLGSLEFLFHTVGFFPIYTSSIGASIIKSKINENKLNIARDKLEIHELKPLETIEISNHSITPFKVSSSLPSSFGFALNTDNGYIVFIDDFIVLNDKNIAFENQLNQIIPKLSDNTLLLITGVGLVGRNSGFTTPKHKSLEQLNRIITPAKGRIFVACYDSNAYSVMTLAQIARMQNRPFIIYSQSFVHLFNTIVRQKLFNNTHLNTISIEEINNSTNSIVVLTSPPDKLYAKLFKIGMNEDERIRYRKSDTFIFMTPKVAGYEEIEAQILDDIARNEVSYYNLGREILSIQASDEDMKFLVSSLKPKYIIPTGGLYRDFINFTMVLKQAGAEQNQILILFNGEVLTIENKKLDSKKNELKLNPKCVDSAGLQEIGASIMFERDQMSESGVVIIIIYFDQKKSEFLNEITYSFLGVSLDVPEKDKLKTKMEELIKKQINDIKDFTTIKKRIGKEISKELKVSIKRAVMNLFTKMTSKAPLILSTIISI \
    --type protein --id S --sequence MVLKTKENKKFDIYLKSSDFAVSKKASKLIKKLNKKHPKRKSLNSFEAKKYDIYFKEVCKAVTNGINNQLICNHINLKILPGEFVVILGKSGSGKTSLLSLISALDRPTSGDSFVCGTNTICCSDAKLTALRNKNVGYIFQQYGLLRDLDVDDNIKLALPLKKRFNNNLEELLERLELKEHRHKKVHKLSGGQQQRVAIARALIKEPKILFGDEPTGAVNIDISKKILQFFVEYNRDKGTTIVIVTHNEKIVELAKRVIKIHDGKIIVDYLNQNPKTIEQINWV \
    multimer_jsons/pools_5k_0040f80.json

Setting name to: pools_5k_0040f80
Write:	/content/multimer_jsons/pools_5k_0040f80.json


In [13]:
# Then fill in data pipeline strings from the monomers
!af3io data-fill \
    --data_dir monomer_msas \
    --json_path multimer_jsons/pools_5k_0040f80.json \
    --output_dir multimer_msas/

Load data index from: monomer_msas/.af3io_data_index.json
Read 18 protein, 0 dna, 0 rna sequence(s)
Data index has 18 protein, 0 dna, 0 rna sequence(s)
Read:	multimer_jsons/pools_5k_0040f80.json
	fill id=B from id=A in /content/monomer_msas/chain_b_data.json
	fill id=C from id=A in /content/monomer_msas/chain_c_data.json
	fill id=D from id=A in /content/monomer_msas/chain_d_data.json
	fill id=E from id=A in /content/monomer_msas/chain_e_data.json
	fill id=F from id=A in /content/monomer_msas/chain_f_data.json
	fill id=G from id=A in /content/monomer_msas/chain_g_data.json
	fill id=H from id=A in /content/monomer_msas/chain_h_data.json
	fill id=I from id=A in /content/monomer_msas/chain_i_data.json
	fill id=J from id=A in /content/monomer_msas/chain_j_data.json
	fill id=K from id=A in /content/monomer_msas/chain_k_data.json
	fill id=L from id=A in /content/monomer_msas/chain_l_data.json
	fill id=M from id=A in /content/monomer_msas/chain_m_data.json
	fill id=N from id=A in /content/mono

In [14]:
# The result should be equal to what was downloaded from zenodo
!md5sum multimer_msas/pools_5k_0040f80_data.json
!md5sum pools_5k_0040f80/pools_5k_0040f80_data.json
#!diff multimer_msas/pools_5k_0040f80_data.json pools_5k_0040f80/pools_5k_0040f80_data.json

211f63d6abcf1c868e7c062d15d4672b  multimer_msas/pools_5k_0040f80_data.json
211f63d6abcf1c868e7c062d15d4672b  pools_5k_0040f80/pools_5k_0040f80_data.json


In [ ]:
# Cleanup
#!rm -rf pools_5k_0040f80.zip
#!rm -rf pools_5k_0040f80
#!rm -rf monomer_jsons
#!rm -rf monomer_msas
#!rm -rf multimer_jsons
#!rm -rf multimer_msas